# DBSCAN Clustering

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [ ]:
# Load the dataset
X = pd.read_csv('finalclusteringdataset.csv')
print(X.head())
print(X.shape)

In [ ]:
# Find optimal epsilon using k-distance graph
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X)
distances, indices = neighbors_fit.kneighbors(X)
distances = np.sort(distances[:, -1], axis=0)

plt.figure(figsize=(10, 6))
plt.plot(distances)
plt.xlabel('Data Points sorted by distance')
plt.ylabel('5-th Nearest Neighbor Distance')
plt.title('K-distance Graph for Epsilon Selection')
plt.grid(True, alpha=0.3)
plt.show()

print(f'Suggested epsilon range: {distances[int(len(distances)*0.95)]:.4f} to {distances[int(len(distances)*0.99)]:.4f}')

In [ ]:
# Apply DBSCAN
eps = 0.5  # Adjust based on k-distance graph
min_samples = 5

dbscan = DBSCAN(eps=eps, min_samples=min_samples)
cluster_labels = dbscan.fit_predict(X)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = list(cluster_labels).count(-1)

print(f'Number of clusters: {n_clusters}')
print(f'Number of noise points: {n_noise}')
print(f'Percentage of noise points: {n_noise/len(X)*100:.2f}%')

In [ ]:
# Evaluate clustering (only if we have at least 2 clusters and not all noise)
if n_clusters > 1 and n_noise < len(X):
    # Filter out noise points for evaluation
    mask = cluster_labels != -1
    if mask.sum() > 0:
        print(f'Silhouette Score: {silhouette_score(X[mask], cluster_labels[mask]):.4f}')
        print(f'Davies-Bouldin Score: {davies_bouldin_score(X[mask], cluster_labels[mask]):.4f}')
else:
    print('Not enough clusters for evaluation')

In [ ]:
# Visualize clusters (2D projection)
from sklearn.decomposition import PCA

# Reduce to 2D for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(10, 7))
# Plot noise points separately
noise_mask = cluster_labels == -1
scatter = plt.scatter(X_pca[~noise_mask, 0], X_pca[~noise_mask, 1], 
                      c=cluster_labels[~noise_mask], cmap='viridis', s=50, alpha=0.6)
plt.scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1], 
           c='red', marker='x', s=100, label='Noise Points')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('DBSCAN Clustering Results')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cluster distribution
unique, counts = np.unique(cluster_labels, return_counts=True)
plt.figure(figsize=(10, 5))
colors = ['red' if u == -1 else 'skyblue' for u in unique]
plt.bar(range(len(unique)), counts, color=colors, edgecolor='black')
plt.xlabel('Cluster')
plt.ylabel('Number of Data Points')
plt.title('DBSCAN Cluster Distribution')
plt.xticks(range(len(unique)), unique)
plt.grid(axis='y', alpha=0.3)
plt.show()

print('\nCluster Distribution:')
for cluster, count in zip(unique, counts):
    if cluster == -1:
        print(f'Noise Points: {count} samples ({count/len(X)*100:.2f}%)')
    else:
        print(f'Cluster {cluster}: {count} samples ({count/len(X)*100:.2f}%)')

In [ ]:
# Save results
X['Cluster'] = cluster_labels
X.to_csv('dbscan_results.csv', index=False)
print('DBSCAN results saved to dbscan_results.csv')